<a href="https://colab.research.google.com/github/Marchielo22/TABELA_FIPE/blob/main/TABELA_FIPE_VEICULOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import time

# 1. Carrega sua planilha de caminhões
df = pd.read_excel('caminhoes.xlsx', sheet_name='Planilha1')

# API URL Base
BASE_URL = "https://parallelum.com.br/fipe/api/v1/caminhoes"

def obter_dados_fipe():
    precos_fipe = []

    # Obter Marcas
    print("Obtendo marcas da FIPE...")
    marcas_res = requests.get(f"{BASE_URL}/marcas").json()
    marcas_dict = {m['nome'].upper(): m['codigo'] for m in marcas_res}

    for idx, row in df.iterrows():
        fabricante = str(row['fabricante']).strip().upper()
        modelo_nome = str(row['modelo']).strip().upper()
        ano = int(row['ano modelo'])

        # 1. Identificar ID da Marca
        marca_id = marcas_dict.get(fabricante)
        if not marca_id:

            marca_id = next((v for k, v in marcas_dict.items() if fabricante in k or k in fabricante), None)

        if not marca_id:
            print(f"Marca {fabricante} não encontrada.")
            precos_fipe.append("Marca não encontrada")
            continue

        # 2. Obter Modelos da Marca
        time.sleep(0.5) # Pausa amigável para evitar bloqueios por excesso de requisições
        modelos_res = requests.get(f"{BASE_URL}/marcas/{marca_id}/modelos").json()
        modelos = modelos_res.get('modelos', [])

        # Encontrar modelo mais parecido
        modelo_id = None
        for m in modelos:
            if modelo_nome in m['nome'].upper() or m['nome'].upper() in modelo_nome:
                modelo_id = m['codigo']
                break

        if not modelo_id:
            print(f"Modelo {modelo_nome} não encontrado na FIPE.")
            precos_fipe.append("Modelo não encontrado")
            continue

        # 3. Obter Anos do Modelo
        time.sleep(0.5)
        anos_res = requests.get(f"{BASE_URL}/marcas/{marca_id}/modelos/{modelo_id}/anos").json()

        # Procurar o código correspondente ao ano desejado (geralmente formato "2012-3" ou "32000" para zero km)
        ano_codigo = None
        for a in anos_res:
            if str(ano) in a['nome']:
                ano_codigo = a['codigo']
                break

        if not ano_codigo:
            print(f"Ano {ano} para o modelo {modelo_nome} não encontrado.")
            precos_fipe.append("Ano não encontrado")
            continue

        # 4. Obter o Preço Final
        time.sleep(0.5)
        valor_res = requests.get(f"{BASE_URL}/marcas/{marca_id}/modelos/{modelo_id}/anos/{ano_codigo}").json()
        preco = valor_res.get('Valor', 'Preço indisponível')
        print(f"Sucesso: {modelo_nome} ({ano}) -> {preco}")
        precos_fipe.append(preco)

    # Salvar resultados
    df['Preço FIPE'] = precos_fipe
    df.to_excel('caminhoes_com_fipe.xlsx', index=False)
    print("Processo concluído! Planilha 'caminhoes_com_fipe.xlsx' gerada.")

if __name__ == "__main__":
    obter_dados_fipe()